###  Free Energy in a Circumplex Model of Emotion — RxInfer.jl
Implementation of [Pattisapu, Verbelen, Pitliya, Kiefer, Albarracin (2024)](https://arxiv.org/abs/2407.02474)


In [ ]:
using Pkg
Pkg.activate(".")
using RxInfer, Distributions, Plots, LinearAlgebra, Random, StableRNGs, ColorSchemes, Tullio
using LogExpFunctions: softmax, xlogx, xlogy, logsumexp 

![](floor_plan_small.png)

In [ ]:
# ─────────────────────────────────────────────────────────────
# §1  GRAPH ENVIRONMENT
# ─────────────────────────────────────────────────────────────
#
#    2 — 1 — 3
#        |
#    5 — 4 — 6
#        |
#    8 — 7 — 9
#        |
#   11 — 10 — 12
#        |
#        13  
#
#  Hubs: node 1, 4, 7, 10
 
const N_LOC  = 13
const P_VIS  = 1.0   # P(visible | co-located with reward)
const EPS    = 1e-12  # numerical floor for probabilities

# ─────────────────────────────────────────────────────────────
# §1.5 ACTION SPACE & TRANSITIONS
# ─────────────────────────────────────────────────────────────

@enum Action NORTH=1 EAST=2 SOUTH=3 WEST=4
const N_ACT = 4

# Rows: Current Node (1-13). Cols: Action (N, E, S, W). 
# Value: Destination Node (agent stays in place if direction is blocked).
const TRANSITION_MAP = [
     1  3  4  2;  # 1  (Hub)
     2  1  2  2;  # 2  (W leaf of 1)
     3  3  3  1;  # 3  (E leaf of 1)
     1  6  7  5;  # 4  (Hub)
     5  4  5  5;  # 5  (W leaf of 4)
     6  6  6  4;  # 6  (E leaf of 4)
     4  9 10  8;  # 7  (Hub)
     8  7  8  8;  # 8  (W leaf of 7)
     9  9  9  7;  # 9  (E leaf of 7)
     7 12 13 11;  # 10 (Hub)
    11 10 11 11;  # 11 (W leaf of 10)
    12 12 12 10;  # 12 (E leaf of 10)
    10 13 13 13   # 13 (S leaf of 10)
]

# ─────────────────────────────────────────────────────────────
# §2  GENERATIVE MODEL MATRICES  (A2, B1, C)
# ─────────────────────────────────────────────────────────────

"""
Build B1: P(next_loc | cur_loc, action)
Returns a (N_LOC, N_LOC, N_ACT) tensor.
"""
function build_B1()
    B1 = zeros(Float64, N_LOC, N_LOC, N_ACT)
    for cur in 1:N_LOC, a in 1:N_ACT
        B1[TRANSITION_MAP[cur, a], cur, a] = 1.0
    end
    return B1
end
 
"""
Build A2: visibility likelihood tensor.
  A2[o_vis, loc, rew_loc] = P(o_vis | loc, rew_loc)
  o_vis ∈ {1=visible, 2=not_visible}.
  With N_rew = N_LOC+1 the extra dimension encodes the 'absent' state,
  which always produces 'not_visible'.
"""
function build_A2(N_rew::Int = N_LOC)
    A2 = zeros(Float64, 2, N_LOC, N_rew)
    for loc in 1:N_LOC, rew in 1:N_rew
        if rew ≤ N_LOC && loc == rew      # Logic for A tensor without `reward absent` belief
            A2[1, loc, rew] = 1.0       # co-located → reward (highly probably) visible
            #A2[2, loc, rew] = 0.0 # not co-located OR absent (when N_rew=N_LOC+1)
        else                              # Logic for A tensor WITH `reward absent` belief
            A2[2, loc, rew] = 1.0
        end
    end
    return A2
end
 

"""
Create reward-to-location mapping matrix.
  Rows    = Physical Locations (N_LOC)
  Columns = Reward States (N_rew)

Supports both:
  1. Standard setup (N_rew == N_LOC) -> Returns a square Identity matrix.
  2. Absent state setup (N_rew == N_LOC + 1) -> Appends a uniform probability column.
"""
function create_reward_to_location_mapping(N_rew::Int = N_LOC + 1)
    # Initialize the rectangular (or square) matrix
    m = zeros(Float64, N_LOC, N_rew)
    
    # 1. Map matching locations (the identity core)
    for i in 1:N_LOC
        m[i, i] = 1.0
    end
    
    # 2. Handle the "absent" state if N_rew includes it
    if N_rew > N_LOC
        # Option 1: Distribute probability uniformly across the 14th column
        m[:, N_rew] .= 1.0 / N_LOC
    end
    
    return m
end

In [ ]:
# A matrix including `reward absent` belief
A_absent = build_A2(14)
absent_reward_to_location = create_reward_to_location_mapping(14);

In [ ]:
B1 = build_B1()
A2 = build_A2()
reward_to_location = create_reward_to_location_mapping(13);

In [ ]:
@model function circumplex_model(reward_observation_tensor, location_transition_tensor, prior_location, prior_reward_location, reward_to_location_mapping, u_prev, T, reward_observation, location_observation)
    old_location ~ Categorical(prior_location)
    reward_location ~ Categorical(prior_reward_location)

    current_location ~ DiscreteTransition(old_location, location_transition_tensor, u_prev)
    location_observation ~ DiscreteTransition(current_location, diageye(13))
    reward_observation ~ DiscreteTransition(current_location, reward_observation_tensor, reward_location)

    previous_location = current_location
    for t in 1:T
        # Step 1 — allocate storage for q_{τ-1} (Bethe beliefs from previous VFE iteration)
        loc_marginalstorage = JointMarginalStorage(Contingency(ones(size(location_transition_tensor))), :location_marginal, t)
        observation_marginalstorage = JointMarginalStorage(Contingency(ones(size(reward_observation_tensor))), :observation_marginal, t)
        # timestep_marginalcollection = JointMarginalCollection([loc_marginalstorage, observation_marginalstorage])

        # Step 2 — Exploration + Ambiguity priors p̃(u_t): computed from stored transition beliefs from previous VMP
        u[t] ~ Exploration(reward_observation) where {meta=loc_marginalstorage}
        location[t] ~ Ambiguity(reward_observation) where {meta=observation_marginalstorage}

        # KL agent
        # u[t] ~ Categorical(fill(0.25, 4))  # KL-Control prior

        # Step 3 — State transition; joint belief q(x_t, x_{t-1}, u_t) is saved into loc_marginalstorage
        location[t] ~ DiscreteTransition(previous_location, location_transition_tensor, u[t]) where {meta=loc_marginalstorage}

        # Step 4 — Simulate future observation; joint q(y_t, x_t) is saved into observation_marginalstorage
        future_rew_obs[t] ~ DiscreteTransition(location[t], reward_observation_tensor, reward_location) where {meta=observation_marginalstorage}
        future_rew_obs[t] ~ Categorical([0.5, 0.5])  # closes the half-edge on the factor graph

        previous_location = location[t]
    end
    if T != 0 # set goal prior as current belief of rew_loc, only skip if no planning horizon
        reward_location_ow ~ OneWay(reward_location)
        location[end] ~ DiscreteTransition(reward_location_ow, reward_to_location_mapping) #diageye(13)
    end
end

In [ ]:
mutable struct JointMarginalStorage{C}
    joint_marginal::C
    name::Symbol # :location_marginal or :observation_marginal
    index::Int   # timestep t
end

function set_marginal!(jmm::JointMarginalStorage{C}, marginal) where {C}
    jmm.joint_marginal = marginal
    return jmm
end

In [ ]:
@marginalrule DiscreteTransition(:out_in_T1) (m_out::Categorical,
                                              m_in::Categorical,
                                              m_T1::Categorical,
                                              q_a::PointMass{<:AbstractArray{T,3}},
                                              meta::JointMarginalStorage) where {T} = begin
    marginal = @call_marginalrule DiscreteTransition(:out_in_T1) (m_out=m_out, m_in=m_in, m_T1=m_T1, q_a=q_a, meta=nothing)
    set_marginal!(meta, marginal)
    return marginal
end

In [ ]:
# rule if predicted obs in planning horizon is using prior_reward_location instead of reward_location variable
@marginalrule DiscreteTransition(:out_in) (m_out::Categorical,
                                            m_in::Categorical,
                                            q_a::PointMass{<:AbstractArray{T,3}},
                                            q_T1::PointMass,
                                            meta::JointMarginalStorage) where {T} = begin
    #println("q_T1::PointMass: ", q_T1)
    #println("Cat(q_T1):", Categorical(BayesBase.getpointmass(q_T1)))
    marginal = @call_marginalrule DiscreteTransition(:out_in) (m_out=m_out, m_in=m_in, m_T1=Categorical(BayesBase.getpointmass(q_T1)), q_a=q_a, meta=nothing)
    #println("with prior_reward_location-PointMass")
    set_marginal!(meta, marginal)
    return marginal
end

In [ ]:
struct OneWay end

@node OneWay Deterministic [out, in]

@rule OneWay(:out, Marginalisation) (m_in::Any,) = m_in

@rule OneWay(:in, Marginalisation) (m_out::Any,) = Uninformative()

@marginalrule OneWay(:out_in) (m_out::Any, m_in::Any) = begin
    return (out=m_in, in=m_in)
end

# added by me (after error during inference showed up)
@marginalrule OneWay(:in) (m_out::DiscreteNonParametric, m_in::DiscreteNonParametric, ) = begin 
    return (in=m_in)
end

@average_energy OneWay (q_out_in::Any,) = begin
    return 0.0
end

In [ ]:
struct Exploration end

@node Exploration Stochastic [out, in]

# Conditional entropy H(X|Y): measures uncertainty in X given Y
# Uses the identity H(X|Y) = H(X,Y) - H(Y), computed elementwise via xlogx/xlogy
function conditional_entropy(x)
    h = sum(x, dims=1)  # marginal over rows q(x_t) → q(x_{t-1}|u_t)
    @tullio res := -(xlogx(x[a,b]) - xlogy(x[a,b], h[b]))
end

@rule Exploration(:out, Marginalisation) (q_in::Any, meta::JointMarginalStorage,) = begin
    q_xt_xprev_given_ut = normalize.(eachslice(components(meta.joint_marginal), dims=3), 1)
    entropies = conditional_entropy.(q_xt_xprev_given_ut)
    return Categorical(softmax(entropies))
end

RxInfer.ReactiveMP.sdtype(any::RxInfer.ReactiveMP.StandaloneDistributionNode) = ReactiveMP.Stochastic() # define the node as stochastic (not deterministic)

@average_energy Exploration (q_out::Any, q_in::Any, meta::Any) = begin
    return 0.0
end

In [ ]:
struct Ambiguity end

@node Ambiguity Stochastic [out, in];

@rule Ambiguity(:out, Marginalisation) (q_in::Any, meta::JointMarginalStorage,) = begin
    slices = normalize.(eachslice(components(meta.joint_marginal), dims=2), 1)
    entropies = entropy.(slices)
    #println("Ambiguity messages: ", softmax(-entropies))
    return Categorical(softmax(-entropies))
end

@average_energy Ambiguity (q_out::Any, q_in::Any, meta::Any) = begin
    return 0.0
end

After standard active inference updates at each timestep $t$, you must calculate the agent's Valence ($V$) and Arousal ($A$):

Valence ($V$) = Utility ($U$) - Expected Utility ($EU$)

$U = \log P(o_t | C)$ (The log-preference of the actual observation received at time $t$).

$EU = \mathbb{E}_{Q(o_t|s_{t-1},\pi)}[\log P(o_t | C)]$ (The expected utility of the predicted observation calculated prior to the actual observation).




<div style="display: flex; justify-content: space-out;">
  <img src="circumplex_model.png" width="40%">
  <img src="circumplex_model_annotated.png" width="40%">
</div>

### Valence
Formula: Utility - Expected Utility
reaches from $+\infin$ (when utility > EU) to 0 (utility meets EU) to $-\infin$ (when utility is smaller than expected utility).
Utility and EU are 0 when reward is predicted or found.
$$\text{Utility}(s_t) = \log p(s_t \mid C) = s_t^\top \log \mathbf{p}(s_T \mid C)$$
$$\text{Utility}(o_t) = \log p(o_t \mid C) = o_t^\top \log \mathbf{p}(o_T \mid C)$$

Since $s_t$ is a one-hot vector, this is simply a lookup into the log-preference vector at the active state 

#### Computing state preference vector $p(s_T|C)$ from the Posterior
Given the DiscreteTransition semantics, the preference vector is a marginalization over reward locations:
$p(s_T \mid C) = \mathbf{M}_r \cdot q(r)$
where $\mathbf{M}_r$ is the reward_to_location_mapping matrix (columns = distributions over states given each reward location) and $q(r)$ is the probability vector of the posterior.

#### Arousal
Arousal ($A$) = Entropy of the posterior

$A = H[Q(s|o)] = \mathbb{E}_{Q(s|o)}[-\log Q(s|o)]$ (Compute the Shannon entropy of the posterior beliefs about the states).
probably the posterior over the reward location in this scenario.

##### The Formula Expanded

$$A = H[Q(s \mid o)] = -\sum_s Q(s \mid o) \cdot \log Q(s \mid o)$$

This measures **how uncertain** the agent is about the state after incorporating observations. High entropy = high arousal = high uncertainty.

##### Arousal Across Certainty Regimes

| Scenario | Q(s\|o) | A | Interpretation |
|---|---|---|---|
| Fully certain | one-hot `[0,1,0,...,0]` | **≈ 0.0** | No arousal, reward location known |
| Fully uncertain | uniform `[1/13, ...]` | **≈ log(13) ≈ 2.56** | Maximum arousal, no information |
| Partial evidence | peaked `[0.05, 0.7, ...]` | **∈ (0, 2.56)** | Moderate arousal |


### Plot both Valence and Arousal on Circumplex model:

Polar Coordinates (Circumplex Mapping)

Radius (Intensity): $r = \sqrt{V^2 + A^2}$

Angle (Emotion type): $\theta = \tan^{-1}(A/V)$

## Iterative Agent Simulation

The agent now navigates the 13-room graph across multiple timesteps. The overall `time_horizon` decrements each step; inference uses a fixed `planning_horizon` until remaining time drops below it, then $T = \min(\text{time\_remaining}, \text{planning\_horizon})$ so both reach 0 together (T-Maze pattern). `infer()` is warm-started from the previous result, the chosen action is executed in the environment, and Valence/Arousal are recorded.

In [ ]:
Base.@kwdef mutable struct CircumplexEnv
    agent_location::Int
    reward_location::Int
end

function onehot(index::Int, states::Int)
    v = zeros(states)
    v[index] = 1.0
    return v
end

create_location_obs(loc::Int) = onehot(loc, N_LOC)

function create_reward_obs(agent_loc::Int, reward_loc::Int)
    agent_loc == reward_loc ? (rand() < P_VIS ? [1.0, 0.0] : [0.0, 1.0]) : [0.0, 1.0]
end

function step_env!(env::CircumplexEnv, action_idx::Int)
    env.agent_location = TRANSITION_MAP[env.agent_location, action_idx]
    return env
end

In [ ]:
Base.@kwdef mutable struct CircumplexBeliefs
    location::Categorical{Float64}
    reward_location::Categorical{Float64}
    action_posterior::Categorical{Float64}
    predicted_next_location::Vector{Float64}
end

peaked(x::Int, n::Int, α=0.999) = (p = fill((1-α)/(n-1), n); p[x] = α; p)

# rew_loc_prior: Int (peaked at that room), :vague, :none (absent state), or a Categorical
function reward_location_belief(rew_loc_prior; reward_absent::Bool=false)
    N_rew = reward_absent ? N_LOC + 1 : N_LOC
    rew_loc_prior isa Categorical && return rew_loc_prior
    rew_loc_prior === :vague && return vague(Categorical, N_rew)
    if rew_loc_prior === :none
        reward_absent || error(":none prior requires reward_absent=true")
        return vague(Categorical, N_rew)
    end
    return Categorical(onehot(rew_loc_prior, N_rew))
end

initialize_beliefs_circumplex(start_loc::Int, rew_loc_prior, prev_action_idx::Int; reward_absent::Bool=false) = CircumplexBeliefs(
    location                 = Categorical(onehot(start_loc, N_LOC)),
    reward_location          = reward_location_belief(rew_loc_prior; reward_absent=reward_absent),
    action_posterior         = Categorical(onehot(prev_action_idx, N_ACT)),
    predicted_next_location  = onehot(TRANSITION_MAP[start_loc, prev_action_idx], N_LOC), # simulate belief about predicted location based on start_loc and prev_action
)

@initialization function efe_tmaze_agent_initialization(prior_location, prior_reward_location, prior_future_locations, T)
    μ(old_location) = prior_location
    μ(reward_location) = prior_reward_location
    if T > 0
        μ(location) = prior_future_locations
    end
end

# Cold start: vague 13-state beliefs
function get_initialization_circumplex(initialization_fn, beliefs, previous_result::Nothing, T)
    return initialization_fn(beliefs.location, beliefs.reward_location, vague(Categorical, N_LOC), T)
end

# Warm start: shift location beliefs forward, carry reward belief over
function get_initialization_circumplex(initialization_fn, beliefs, previous_result, T)
    current_location_belief = last(previous_result.posteriors[:location])[1]
    future_location_beliefs = last(previous_result.posteriors[:location])[2:end]
    reward_location_belief  = last(previous_result.posteriors[:reward_location])
    return initialization_fn(current_location_belief, reward_location_belief, future_location_beliefs, T)
end

In [ ]:
# Planning depth: fixed at planning_horizon until time_remaining drops below it, then both shrink to 0
effective_planning_horizon(time_remaining, planning_horizon) = min(time_remaining, planning_horizon)

function execute_step_circumplex(
        env, beliefs, config, time_remaining,
        previous_result, previous_action_idx; initialization_fn, reward_absent::Bool=false)

    A_mat, rew_to_loc = if reward_absent
        (A_absent, absent_reward_to_location)
    else
        (A2, reward_to_location)
    end

    T = effective_planning_horizon(time_remaining, config.planning_horizon)
    initialization = get_initialization_circumplex(initialization_fn, beliefs, previous_result, T)
    result = infer(
        model = circumplex_model(
            reward_observation_tensor  = A_mat,
            location_transition_tensor = B1,
            prior_location             = probvec(beliefs.location),
            prior_reward_location      = probvec(beliefs.reward_location),
            reward_to_location_mapping = rew_to_loc,
            u_prev                     = onehot(previous_action_idx, N_ACT),
            T                          = T
        ),
        data        = (location_observation = create_location_obs(env.agent_location),
                       reward_observation   = create_reward_obs(env.agent_location, env.reward_location)),
        options     = (force_marginal_computation = true,),
        iterations  = config.n_iterations,
        #free_energy = true,
        initialization = initialization
    )

    if time_remaining > 0 # no planning loop for last step, thus no :u key
        next_action_idx = Int(mode(first(last(result.posteriors[:u]))))
        beliefs.action_posterior = first(last(result.posteriors[:u]))
    else
        next_action_idx = previous_action_idx
    end

    beliefs.location        = last(result.posteriors[:current_location])
    beliefs.reward_location = last(result.posteriors[:reward_location])

    return next_action_idx, result
end

In [ ]:
using LinearAlgebra: dot

function run_circumplex_episode(start_loc, reward_loc_env, reward_loc_prior, config;
                                initialization_fn, reward_absent::Bool=false)
    env = CircumplexEnv(agent_location=start_loc, reward_location=reward_loc_env)
    
    previous_result = nothing
    previous_action_idx = Int(NORTH) # Consider passing NORTH or fetching from env
    
    beliefs = initialize_beliefs_circumplex(
        start_loc, reward_loc_prior, previous_action_idx;
        reward_absent=reward_absent
    )

    rew_to_loc = reward_absent ? absent_reward_to_location : reward_to_location

    # Pre-allocate output arrays if config.time_horizon is known
    horizon = config.time_horizon
    valences    = Vector{Float64}(undef, horizon + 1)
    arousals    = Vector{Float64}(undef, horizon + 1)
    locations   = Vector{Int}(undef, horizon + 1)
    actions     = Vector{Int}(undef, horizon + 1)
    rew_beliefs = Vector{Vector{Float64}}()
    sizehint!(rew_beliefs, horizon + 1)

    # Use an explicit loop index to track array insertion
    step_idx = 1 

    for time_remaining in horizon:-1:0
        next_action_idx, result = execute_step_circumplex(
            env, beliefs, config, time_remaining, previous_result, previous_action_idx;
            initialization_fn=initialization_fn, reward_absent=reward_absent
        )
        
        # Calculate Current Posterior
        reward_posterior = probvec(last(result.posteriors[:reward_location]))
        
        # Calculate Expected Utility (EU)
        if isempty(rew_beliefs)
            # First round: use current posterior
            preference = rew_to_loc * reward_posterior
        else
            # Subsequent rounds: use beliefs from previous step
            preference = rew_to_loc * rew_beliefs[end]
        end
        
        # Fuse broadcasting to prevent intermediate allocations
        log_pref = @. log(preference + eps()) 
        EU = dot(beliefs.predicted_next_location, log_pref)

        # Calculate actual utility and valence
        preference_current = rew_to_loc * reward_posterior
        log_pref_current   = @. log(preference_current + eps())
        
        obs_reward, _ = create_reward_obs(env.agent_location, env.reward_location)
        utility = log(obs_reward + 50000 * eps())
        valence = utility - EU
        
        # Calculate Arousal
        arousal = entropy(last(result.posteriors[:reward_location]))

        # Store metrics (avoiding push! by indexing directly where possible)
        valences[step_idx]  = valence
        arousals[step_idx]  = arousal
        locations[step_idx] = env.agent_location
        actions[step_idx]   = next_action_idx
        push!(rew_beliefs, copy(reward_posterior))

        if env.agent_location == env.reward_location
            @info "Reached reward location $(env.reward_location) at step $(step_idx)!"
            # Resize arrays to actual length if broken early
            resize!(valences, step_idx)
            resize!(arousals, step_idx)
            resize!(locations, step_idx)
            resize!(actions, step_idx)
            break
        end

        # Prepare for next step
        previous_result = result
        previous_action_idx = next_action_idx
        step_env!(env, next_action_idx)
        
        if time_remaining > 0
            @debug "env.agent_location: $(env.agent_location)"
            beliefs.predicted_next_location = onehot(env.agent_location, N_LOC) 
        end
        
        step_idx += 1
    end

    # Use stack for cleaner, faster array transposition (Julia 1.9+)
    # If on an older version, revert to: reduce(hcat, rew_beliefs)'
    rew_matrix = stack(rew_beliefs; dims=1) 

    return valences, arousals, rew_matrix, locations, actions
end

config_circumplex = (time_horizon = 80, planning_horizon = 6, n_iterations = 20)
# Scenario 1: vague prior; object present
valences, arousals, rew_beliefs, locations, actions = run_circumplex_episode(1, 9, :vague, config_circumplex;
    initialization_fn = efe_tmaze_agent_initialization, reward_absent = false)

# Scenario 2: precise,correct prior; object present
# valences, arousals, rew_beliefs, locations, actions = run_circumplex_episode(1, 11, 11, config_circumplex;
#     initialization_fn = efe_tmaze_agent_initialization, reward_absent = false)

# Scenario 3: precise,incorrect prior; object present
# reward_loc_env=9 (true reward), reward_loc_prior=5 (misleading belief)
# valences, arousals, rew_beliefs, locations, actions = run_circumplex_episode(1, 9, 5, config_circumplex;
#     initialization_fn = efe_tmaze_agent_initialization, reward_absent = false)

# Scenario 4: vague prior, maybe here; object not-present
# use reward_loc_env > N_LOC to initalize an env with absent reward.
# valences, arousals, rew_beliefs, locations, actions = run_circumplex_episode(1, 15, :vague, config_circumplex;
#     initialization_fn = efe_tmaze_agent_initialization, reward_absent = true)

# Scenario 5: vague prior, definitely here; object not-present
# valences, arousals, rew_beliefs, locations, actions = run_circumplex_episode(1, 15, :vague, config_circumplex;
#     initialization_fn = efe_tmaze_agent_initialization, reward_absent = false)
println("Location path: ", locations)


In [ ]:
# ── Shared plotting helper ────────────────────────────────────────────────
using Plots.PlotMeasures # needed for margin
const _ACTION_SHAPE = Dict(1 => :utriangle, 2 => :rtriangle, 3 => :dtriangle, 4 => :ltriangle)
const _ACTION_LABEL = Dict(1 => "N↑", 2 => "E→", 3 => "S↓", 4 => "W←")
const _ACTION_COLOR = Dict(1 => "#4daf4a", 2 => "#377eb8", 3 => "#ff7f00", 4 => "#984ea3")
const _CIRCUMPLEX_EMOTION_LABELS = (
    ("Happy", 0), ("Excited", π/4), ("Alert", π/2), ("Angry", 3π/4),
    ("Sad", π), ("Depressed", 5π/4), ("Calm", 3π/2), ("Relaxed", 7π/4),
)

"""Valence/arousal → circumplex polar coordinates (same normalization as `plot_episode`)."""
function _circumplex_coords(valences, arousals)
    arousal_midpoint = log(N_LOC) / 2
    v_norm = clamp.(valences ./ log(N_LOC), -1.0, 1.0)
    a_norm = clamp.((arousals .- arousal_midpoint) ./ arousal_midpoint, -1.0, 1.0)
    θs = atan.(a_norm, v_norm)
    rs = clamp.(sqrt.(v_norm.^2 .+ a_norm.^2), 0.0, 1.0)
    return θs, rs
end

function _circumplex_polar_plot(title::String, θs, rs, step_indices, circ_cmap; highlight_last::Bool=false)
    t_end = isempty(step_indices) ? 0 : step_indices[end]
    title_suffix = length(step_indices) <= 1 ? "step $t_end" : "steps 1…$t_end"

    Plots.gr_cbar_width[] = 0.030

    p_circ = plot(
        proj = :polar, legend = false, grid = true, ylims = (0, 1.0),
        title = "$(title) — Emotion Trajectory ($title_suffix)")
    yticks!(p_circ, [0.25, 0.5, 0.75, 1.0], string.([0.25, 0.5, 0.75, 1.0]))
    
    if !isempty(step_indices)
        msizes = highlight_last ? [s == t_end ? 10 : 5 for s in step_indices] : 5
        
        scatter!(
            p_circ, θs[step_indices], rs[step_indices],
            markersize = msizes,
            zcolor = step_indices,
            color = circ_cmap,
            colorbar = true,
            colorbar_title = "Step",
            colorbar_titlefont = font(8),
            guidefontsize = 7, 
            alpha = 0.85,
            markerstrokewidth = 0.7,
            markerstrokecolor = :black,
            series_annotations = [text(string(s), 1, :center, :black) for s in step_indices]
        )
        
        plot!(p_circ, θs[step_indices], rs[step_indices], lw = 1, alpha = 0.35, color = :gray, label = "")
    end
    
    for (label, angle) in _CIRCUMPLEX_EMOTION_LABELS
        r_anchor = 1.15
        x_cart = r_anchor * cos(angle)
        y_cart = r_anchor * sin(angle)
        
        y_final = sin(angle) > 0.1 ? y_cart + 0.12 : y_cart - 0.12
        annotate!(p_circ, x_cart, y_final, text(label, 8, :center, :black))
    end
    
    return p_circ
end

function plot_episode(title, valences, arousals, rew_beliefs, locations, actions;
                      reward_loc, save_prefix)
    """Timeseries + circumplex polar plot, then reward-belief heatmap."""
    rew_beliefs = Matrix(rew_beliefs)  # accept Adjoint from reduce(hcat, ...)'
    steps = 1:length(valences)
    arousal_midpoint = log(N_LOC) / 2

    p_ts = plot(steps, valences, label="Valence", lw=2, color=:steelblue,
                xlabel="Step", ylabel="Value (nats)",
                title="$(title) — Valence & Arousal over Episode")
    plot!(p_ts, steps, arousals, label="Arousal", lw=2, ls=:dash, color=:firebrick)
    hline!(p_ts, [0], color=:black, lw=0.5, ls=:dot, label="")

    v_norm = clamp.(valences ./ log(N_LOC), -1.0, 1.0)
    a_norm = clamp.((arousals .- arousal_midpoint) ./ arousal_midpoint, -1.0, 1.0)
    θs = atan.(a_norm, v_norm)
    rs = clamp.(sqrt.(v_norm.^2 .+ a_norm.^2), 0.0, 1.0)

    circ_cmap = cgrad([get(ColorSchemes.viridis, x) for x in range(0.32, 0.96, length=64)])

    # Just call the shared function with the full range of steps!
    p_circ = _circumplex_polar_plot(title, θs, rs, steps, circ_cmap, highlight_last=false)

    p_top = plot(p_ts, p_circ, layout=(1, 2), size=(950, 420), dpi=150, margin=7Plots.mm)   
    savefig(p_top, "$(save_prefix)_circumplex.png")
    display(p_top)

    n_steps = size(rew_beliefs, 1)  # rew_beliefs[step, room]
    vmax = maximum(rew_beliefs)
    # Plots: rows → y (room), cols → x (step); same layout as Python imshow(rew_beliefs.T)
    p_hm = heatmap(rew_beliefs';
        xlabel="Step", ylabel="Room (1-indexed)",
        yticks=(1:N_LOC, string.(1:N_LOC)),
        title="$(title) — Reward-Location Belief Dynamics Q(s^rew_t)\nOriented markers show agent position and action at each step",
        colorbar_title="Belief probability", clims=(0.0, vmax), color=:blues, size=(1000, 450), dpi=150)
    hline!(p_hm, [reward_loc], color=:red, lw=1.5, ls=:dash,
           label="Reward room $(reward_loc)")

    plotted = Set{Int}()
    for (t, (loc, act)) in enumerate(zip(locations, actions))
        lbl = act in plotted ? "" : _ACTION_LABEL[act]
        scatter!(p_hm, [t], [loc], shape=_ACTION_SHAPE[act], markersize=8,
                 markercolor=_ACTION_COLOR[act], markerstrokewidth=0.5,
                 markerstrokecolor=:black, label=lbl)
        push!(plotted, act)
    end

    savefig(p_hm, "$(save_prefix)_reward_belief.png")
    display(p_hm)
end

plot_episode(
    "VMP", valences, arousals, rew_beliefs, locations, actions,
    reward_loc=6, save_prefix="vmp",
)

In [ ]:
using Plots

# 1. Define explicit (x, y) coordinates for nodes 1 to 13
const NODE_COORDS = [
    ( 0.0,  3.0), # 1  (Hub)
    (-1.0,  3.0), # 2  (W leaf)
    ( 1.0,  3.0), # 3  (E leaf)
    ( 0.0,  2.0), # 4  (Hub)
    (-1.0,  2.0), # 5  (W leaf)
    ( 1.0,  2.0), # 6  (E leaf)
    ( 0.0,  1.0), # 7  (Hub)
    (-1.0,  1.0), # 8  (W leaf)
    ( 1.0,  1.0), # 9  (E leaf)
    ( 0.0,  0.0), # 10 (Hub)
    (-1.0,  0.0), # 11 (W leaf)
    ( 1.0,  0.0), # 12 (E leaf)
    ( 0.0, -1.0)  # 13 (S leaf)
]

# Reward-absent belief state (Scenario 4: N_rew = N_LOC + 1, index 14)
const REWARD_ABSENT_IDX = N_LOC + 1
const ABSENT_NODE_COORD = (0.0, -2.0)   # below room 13
const ABSENT_ATTACH_LOC = 10            # dashed link to hub above room 13

# 2. Define the unique undirected connections (edges)
const EDGES = [
    (1, 2), (1, 3), (1, 4),
    (4, 5), (4, 6), (4, 7),
    (7, 8), (7, 9), (7, 10),
    (10, 11), (10, 12), (10, 13)
]

In [ ]:
# ── Animated floor plan + circumplex polar plot ───────────────────────────

"""True when belief matrix includes the reward-absent state (column/row 14)."""
_has_absent_belief(rew_beliefs) = size(Matrix(rew_beliefs), 2) >= REWARD_ABSENT_IDX

"""Draw static graph edges and room labels; optional ∅ node for Scenario 4."""
function _floorplan_graph_base(; show_absent::Bool = false)
    y_lo = show_absent ? -2.5 : -1.5
    p = plot(
        legend = false,
        ticks = false,
        showaxis = false,
        grid = false,
        aspect_ratio = :equal,
        xlims = (-1.5, 1.5),
        ylims = (y_lo, 3.5),
    )
    for (src, dst) in EDGES
        x1, y1 = NODE_COORDS[src]
        x2, y2 = NODE_COORDS[dst]
        plot!(p, [x1, x2], [y1, y2], color = :black, lw = 2, z_order = :back)
    end
    xs = [c[1] for c in NODE_COORDS]
    ys = [c[2] for c in NODE_COORDS]
    labels = [(xs[i], ys[i], text(string(i), 10, :center, :black, :bold)) for i in 1:N_LOC]
    annotate!(p, labels)
    if show_absent
        ax, ay = ABSENT_NODE_COORD
        hx, hy = NODE_COORDS[ABSENT_ATTACH_LOC]
        plot!(p, [hx, ax], [hy, ay], color = :gray, lw = 1.5, ls = :dash, z_order = :back)
        annotate!(p, ax, ay, text("∅", 11, :center, :dimgray, :bold))
        annotate!(p, ax, ay - 0.35, text("absent", 7, :center, :gray60))
    end
    return p, xs, ys
end

"""
Animate agent movement on the abstract floor plan and circumplex emotion trajectory.

Left panel: graph with node fill ∝ Q(s^rew = room) at step `t`, path trace, and
action-shaped marker at the current room. Right panel: circumplex polar plot (steps 1…`t`).
"""
function animate_episode_floorplan(
    title::String,
    valences,
    arousals,
    rew_beliefs,
    locations::AbstractVector{Int},
    actions::AbstractVector{Int};
    reward_loc::Union{Int, Nothing} = nothing,
    save_path::String = "episode_floorplan.gif",
    fps::Int = 2,
)
    rew_beliefs = Matrix(rew_beliefs)
    valences = collect(valences)
    arousals = collect(arousals)
    has_absent = _has_absent_belief(rew_beliefs)
    n_steps = size(rew_beliefs, 1)
    n_steps == length(locations) == length(actions) == length(valences) == length(arousals) ||
        error("episode arrays must have equal length (got $n_steps belief steps)")

    vmax = maximum(rew_beliefs)
    vmax = vmax > 0 ? vmax : 1.0
    bel_cmap = cgrad(:blues)
    circ_cmap = cgrad([get(ColorSchemes.viridis, x) for x in range(0.32, 0.96, length = 64)])
    θs, rs = _circumplex_coords(valences, arousals)

    anim = @animate for t in 1:n_steps
        bel = rew_beliefs[t, :]
        bel_rooms = bel[1:N_LOC]
        bel_absent = has_absent ? bel[REWARD_ABSENT_IDX] : 0.0
        hub_mask = [i in (1, 4, 7, 10) for i in 1:N_LOC]

        p_fp, xs, ys = _floorplan_graph_base(; show_absent = has_absent)
        scatter!(
            p_fp, xs, ys,
            markersize = [hub_mask[i] ? 22 : 18 for i in 1:N_LOC],
            zcolor = bel_rooms,
            color = bel_cmap,
            clims = (0.0, vmax),
            colorbar = true,
            colorbar_title = "Belief",
            markerstrokecolor = :black,
            markerstrokewidth = 2,
            label = "",
        )

        if has_absent
            ax, ay = ABSENT_NODE_COORD
            scatter!(
                p_fp, [ax], [ay],
                markersize = 22,
                zcolor = [bel_absent],
                color = bel_cmap,
                clims = (0.0, vmax),
                markerstrokecolor = :dimgray,
                markerstrokewidth = 2,
                label = "Q(absent) = $(round(bel_absent, digits=3))",
            )
        end

        if !isnothing(reward_loc) && 1 ≤ reward_loc ≤ N_LOC
            rx, ry = NODE_COORDS[reward_loc]
            scatter!(
                p_fp, [rx], [ry],
                markersize = 26,
                markercolor = :transparent,
                markerstrokecolor = :red,
                markerstrokewidth = 2.5,
                label = "Reward room $reward_loc",
            )
        end

        if t > 1
            for s in 1:(t - 1)
                loc_a, loc_b = locations[s], locations[s + 1]
                loc_a == loc_b && continue
                x1, y1 = NODE_COORDS[loc_a]
                x2, y2 = NODE_COORDS[loc_b]
                plot!(p_fp, [x1, x2], [y1, y2], color = :gray, lw = 1.5, alpha = 0.55, label = "")
            end
        end

        loc = locations[t]
        act = actions[t]
        ax, ay = NODE_COORDS[loc]
        scatter!(
            p_fp, [ax], [ay],
            shape = _ACTION_SHAPE[act],
            markersize = 14,
            markercolor = _ACTION_COLOR[act],
            markerstrokewidth = 1.2,
            markerstrokecolor = :black,
            label = "Step $t: room $loc, $(_ACTION_LABEL[act])",
        )

        absent_note = has_absent ? "; ∅ = reward absent" : ""
        plot!(
            p_fp,
            title = "$(title) — Step $t / $n_steps\nNode color = Q(s^rew = room)$(absent_note)",
        )

        p_circ = _circumplex_polar_plot(title, θs, rs, 1:t, circ_cmap, highlight_last=true)
        plot(
            p_fp, p_circ,
            layout = (1, 2),
            size = (1150, 480),
            dpi = 120,
            margin = 7Plots.mm
        )
    end

    gif(anim, save_path; fps = fps)
    display(anim)
    return anim
end

# Requires: episode variables from run_circumplex_episode, NODE_COORDS/EDGES, _ACTION_*
# Scenario 4 (reward_absent=true): rew_beliefs has 14 columns; ∅ node appears automatically.
# reward_loc: room 1–13, REWARD_ABSENT_IDX (14) when env reward is absent, or `nothing`
reward_loc_anim = if _has_absent_belief(rew_beliefs)
    REWARD_ABSENT_IDX   # Scenario 4: highlight ∅ row when env uses code > N_LOC
elseif 15 > N_LOC
    nothing
else
    15
end
animate_episode_floorplan(
    "VMP",
    valences,
    arousals,
    rew_beliefs,
    locations,
    actions;
    reward_loc = reward_loc_anim,
    save_path = "vmp_floorplan_animation.gif",
    fps = 2,
)